# SupeRR Selector Season 2 — Phase 2 Comprehensive Dataset Audit

**Scope:** Read-only structural inspection, schema validation, competition separation, match reconciliation, delivery-level checks, and player-mapping validation across all six raw competition CSVs in `data/raw/Data Pack/`.

**Guardrails:**
- Raw CSV files are never modified.
- All paths are resolved relative to the `SupeRR_Selector/` project root.
- No pressure metrics, cleaning transformations, or player rankings are computed in this notebook.

In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

# Resolve project root dynamically whether launched from project root or notebooks/
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "data" / "raw").is_dir() else cwd.parent
DATA_PACK_DIR = ROOT / "data" / "raw" / "Data Pack"
OUT_DIR = ROOT / "outputs"

assert DATA_PACK_DIR.is_dir(), f"Data Pack directory not found at {DATA_PACK_DIR}"
print(f"Project root : {ROOT.name}")
print(f"Data source  : {DATA_PACK_DIR.relative_to(ROOT)}")
print(f"Audit outputs: {OUT_DIR.relative_to(ROOT)}")

## 1. Load Generated Phase 2 Audit Outputs & Verify Raw Tables

Run `python src/02_dataset_audit.py` from the project root before viewing the audit tables below, or execute the cell to inspect both the raw files and the generated CSV audit reports.

In [ ]:
dq_report = pd.read_csv(OUT_DIR / "data_quality_report.csv")
comp_summary = pd.read_csv(OUT_DIR / "competition_summary.csv")
match_recon = pd.read_csv(OUT_DIR / "match_reconciliation.csv")
missing_report = pd.read_csv(OUT_DIR / "missing_values_report.csv")
player_report = pd.read_csv(OUT_DIR / "player_mapping_report.csv")
delivery_report = pd.read_csv(OUT_DIR / "delivery_validation_report.csv")
innings_summary = pd.read_csv(OUT_DIR / "innings_summary.csv")
col_dict = pd.read_csv(OUT_DIR / "column_dictionary.csv")

display(dq_report)

## 2. Missing Values, Literal `'None'` Tokens, and Zero Sentinels

Notice the distinction between true blank cells in the raw CSVs and literal string `'None'` values (notably 189,167 rows in `extras_type`), which default `pandas.read_csv()` silently converts to `NaN`.

In [ ]:
flagged_missing = missing_report[
    (missing_report["raw_blank_count"] > 0)
    | (missing_report["literal_none_count"] > 0)
    | (missing_report["zero_sentinel_count"] > 0)
]
display(flagged_missing)

## 3. Competition Coverage: IPL vs. T20I vs. Non-IPL T20

The official Round 1 brief specifies historical **IPL and T20I** data from 2022 onwards. The raw data pack also contains **2 non-IPL T20 competitions** (`Mens T20 Emerging Teams Asia Cup 2024` and `Asia Cup Rising Stars 2025`, totaling 30 matches / 6,985 deliveries), which are explicitly separated below.

In [ ]:
cat_totals = (
    comp_summary.groupby("eligibility_category", dropna=False)
    .agg(
        competitions=("comp_id", "nunique"),
        matches_metadata=("matches_in_metadata", "sum"),
        matches_ball_data=("matches_with_ball_data", "sum"),
        deliveries=("total_deliveries", "sum"),
        legal_deliveries=("legal_deliveries", "sum"),
    )
    .reset_index()
)
display(cat_totals)

display(
    comp_summary[comp_summary["eligibility_category"].isin(["IPL", "Non-IPL T20"])][
        ["comp_id", "comp_name", "match_format", "season", "eligibility_category",
         "matches_in_metadata", "matches_with_ball_data", "total_deliveries",
         "min_match_date", "max_match_date"]
    ]
)

In [ ]:
# Exploratory static audit chart using RR PERFORMANCE LAB palette
fig, ax = plt.subplots(figsize=(8.5, 4.2), facecolor="#24172C")
ax.set_facecolor("#392940")

colors = {"IPL": "#E83E80", "T20I": "#D8B45A", "Non-IPL T20": "#B8A6C2"}
bar_colors = [colors.get(c, "#7C9EF2") for c in cat_totals["eligibility_category"]]
bars = ax.bar(cat_totals["eligibility_category"], cat_totals["legal_deliveries"], color=bar_colors, width=0.52)

ax.set_title("Legal Deliveries by Competition Eligibility Category", color="#F5F1F5", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Legal Deliveries", color="#F5F1F5", fontsize=10)
ax.tick_params(colors="#F5F1F5")
ax.grid(axis="y", color="#B8A6C2", alpha=0.18, linestyle="--")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("#B8A6C2")
    ax.spines[spine].set_alpha(0.35)

for bar, val in zip(bars, cat_totals["legal_deliveries"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + 1500,
        f"{int(val):,}",
        ha="center",
        va="bottom",
        color="#F5F1F5",
        fontsize=10,
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

## 4. Match Reconciliation & Innings Completeness

In [ ]:
display(match_recon)
print("\nInnings status breakdown:")
display(innings_summary["innings_status_flag"].value_counts().rename_axis("status_flag").to_frame("innings_count"))

## 5. Delivery-Level Validation & DuckDB Cross-Check

Verify runs, extras (`wide`, `Overstep`, `other`, `byes`, `legbyes`, `Penalty`), legal delivery logic, wicket consistency, and `bowl_speed` sentinels.

In [ ]:
display(delivery_report)

# Independent DuckDB verification directly on raw ball_by_ball.csv
con = duckdb.connect(":memory:")
balls_path = str(DATA_PACK_DIR / "ball_by_ball.csv").replace("\\", "/")
extras_check = con.execute(f"""
    SELECT
        extras_type,
        COUNT(*) AS deliveries,
        SUM(CAST(runs AS BIGINT)) AS batter_runs,
        SUM(CAST(extras1_runs AS BIGINT)) AS extras1_runs,
        SUM(CAST(extras_2_runs AS BIGINT)) AS extras2_runs
    FROM read_csv('{balls_path}', header=true, nullstr='')
    GROUP BY extras_type
    ORDER BY deliveries DESC
""").df()
display(extras_check)

## 6. Player Mapping & Identifier Integrity

In [ ]:
display(player_report)